### Imports

In [17]:
import os
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

### Load observations

In [18]:
from pathlib import Path
required = [f"function_{i}.csv" for i in range(1, 9)]
roots = [Path.cwd(), *Path.cwd().parents]
folders = [Path("Data"), Path("7. Module 18") / "Data", Path("7. Module 18") / "updated_data_week_6", Path("6. Module 17") / "updated_data_week_6", Path("updated_data_week_6"), Path("7. Capstone Project") / "6. Module 17" / "updated_data_week_6"]
data_folder = next((root / folder for root in roots for folder in folders if all((root / folder / f).is_file() for f in required)), None)
if data_folder is None:
    raise FileNotFoundError("Place function_1.csv through function_8.csv in a Data or updated_data_week_6 folder.")
data = {i: pd.read_csv(data_folder / f"function_{i}.csv") for i in range(1, 9)}


display(pd.DataFrame([{"Function": i, "Observations": len(data[i])} for i in range(1, 9)]).set_index("Function"))

,Observations
Function,
1,16
2,16
3,21
4,36
5,26
6,26
7,36
8,46


### Week 6 query results

In [19]:
week_6_points = {1: [0.773956, 0.438878], 2: [0.729513, 0.111999], 3: [0.30826, 0.310001, 0.479247], 4: [0.329887, 0.354827, 0.355151, 0.453774], 5: [0.939869, 0.992261, 0.978815, 0.999723], 6: [0.397519, 0.393747, 0.471443, 0.71138, 0.165385], 7: [0.174885, 0.324058, 0.479146, 0.269144, 0.331135, 0.61449], 8: [0.042609, 0.051269, 0.314729, 0.12723, 0.965221, 0.369, 0.150066, 0.389171]}
week_6_outputs = {1: 9.656222973626152e-41, 2: 0.4718443083527728, 3: -0.015498449969482334, 4: -0.20043060928325174, 5: 7121.1519545556275, 6: -0.37973950049527283, 7: 2.9293698997248208, 8: 9.8405301677974}

### Evaluate previous queries

In [20]:
week_6_results = {}
for i in range(1, 9):
    previous_best = data[i]['output'].max()
    new_output = week_6_outputs[i]
    improvement = new_output - previous_best
    is_new_best = new_output > previous_best
    week_6_results[i] = {'previous_best': previous_best, 'new_output': new_output, 'improvement': improvement, 'new_best': is_new_best}

display(pd.DataFrame.from_dict(week_6_results, orient="index").rename_axis("Function"))

,previous_best,new_output,improvement,new_best
Function,,,,
1,7.710875e-16,9.656223e-41,-7.710875e-16,False
2,7.469623e-01,4.718443e-01,-2.751179e-01,False
3,-1.549845e-02,-1.549845e-02,-3.469447e-17,False
4,-2.004306e-01,-2.004306e-01,-2.775558e-17,False
5,7.121152e+03,7.121152e+03,-9.094947e-13,False
6,-3.797395e-01,-3.797395e-01,-5.551115e-17,False
7,2.929370e+00,2.929370e+00,0.000000e+00,False
8,9.911249e+00,9.840530e+00,-7.071921e-02,False


### Update observations

In [21]:
for i in range(1, 9):
    point = np.asarray(week_6_points[i], dtype=float)
    X = data[i].drop(columns="output").to_numpy(dtype=float)
    if not np.any(np.all(np.isclose(X, point, atol=1e-9, rtol=0), axis=1)):
        new_row = {f"x{j+1}": value for j, value in enumerate(point)}
        new_row["output"] = week_6_outputs[i]
        data[i] = pd.concat([data[i], pd.DataFrame([new_row])], ignore_index=True)


### Gaussian Process models

In [22]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
import warnings
from sklearn.exceptions import ConvergenceWarning
gp_models = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    kernel = RBF(length_scale=1.0, length_scale_bounds=(1e-05, 100000.0))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y)
    gp_models[i] = gp

display(pd.DataFrame({"Function": list(gp_models), "Kernel": [str(gp_models[i].kernel_) for i in gp_models]}).set_index("Function"))

,Kernel
Function,
1,RBF(length_scale=1e-05)
2,RBF(length_scale=0.0431)
3,RBF(length_scale=0.193)
4,RBF(length_scale=0.436)
5,RBF(length_scale=0.363)
6,RBF(length_scale=0.558)
7,RBF(length_scale=0.3)
8,RBF(length_scale=1.02)


### GP prediction diagnostics

In [23]:
rng = np.random.default_rng(42)
diagnostic_candidates = {}
gp_diagnostics = {}
for i in range(1, 9):
    n_dimensions = data[i].shape[1] - 1
    candidates = rng.random((100000, n_dimensions))
    diagnostic_candidates[i] = candidates
    mean_pred, std_pred = gp_models[i].predict(candidates, return_std=True)
    gp_diagnostics[i] = {'mean': mean_pred, 'std': std_pred}

### UCB parameter tuning

In [24]:
ucb_kappas = [0.5, 1.0, 1.5, 2.5]
ucb_tuning_results = {}
for i in range(1, 9):
    candidates = diagnostic_candidates[i]
    mean_pred = gp_diagnostics[i]['mean']
    std_pred = gp_diagnostics[i]['std']
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    best_index = np.argmax(y_existing)
    best_point = X_existing[best_index]
    best_output = y_existing[best_index]
    ucb_tuning_results[i] = {}
    for kappa in ucb_kappas:
        ucb_scores = mean_pred + kappa * std_pred
        idx = np.argmax(ucb_scores)
        point = candidates[idx]
        predicted = mean_pred[idx]
        uncertainty = std_pred[idx]
        score = ucb_scores[idx]
        distance_best = np.linalg.norm(point - best_point)
        distances_existing = np.linalg.norm(X_existing - point, axis=1)
        distance_nearest = distances_existing.min()
        ucb_tuning_results[i][kappa] = {'point': point, 'predicted': predicted, 'uncertainty': uncertainty, 'score': score, 'distance_best': distance_best, 'distance_nearest': distance_nearest}

### Expected Improvement tuning

In [25]:
from scipy.stats import norm
xi_proportions = [0.0, 0.01, 0.05, 0.1]
ei_tuning_results = {}
for i in range(1, 9):
    candidates = diagnostic_candidates[i]
    mean_pred = gp_diagnostics[i]['mean']
    std_pred = gp_diagnostics[i]['std']
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    best_index = np.argmax(y_existing)
    best_point = X_existing[best_index]
    best_output = y_existing[best_index]
    y_std = np.std(y_existing)
    ei_tuning_results[i] = {}
    for xi_prop in xi_proportions:
        xi = xi_prop * y_std
        improvement = mean_pred - best_output - xi
        with np.errstate(divide='ignore', invalid='ignore'):
            Z = np.divide(improvement, std_pred, out=np.zeros_like(improvement), where=std_pred > 0)
            ei_scores = improvement * norm.cdf(Z) + std_pred * norm.pdf(Z)
        ei_scores[std_pred <= 0] = 0.0
        idx = np.argmax(ei_scores)
        point = candidates[idx]
        predicted = mean_pred[idx]
        uncertainty = std_pred[idx]
        score = ei_scores[idx]
        distance_best = np.linalg.norm(point - best_point)
        distances_existing = np.linalg.norm(X_existing - point, axis=1)
        distance_nearest = distances_existing.min()
        ei_tuning_results[i][xi_prop] = {'xi': xi, 'point': point, 'predicted': predicted, 'uncertainty': uncertainty, 'score': score, 'distance_best': distance_best, 'distance_nearest': distance_nearest}

### Neural-network ensembles

In [26]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np
nn_models = {}
nn_x_scalers = {}
nn_y_scalers = {}
nn_training_losses = {}
n_ensemble = 5
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float).reshape(-1, 1)
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    X_scaled = x_scaler.fit_transform(X)
    y_scaled = y_scaler.fit_transform(y).ravel()
    models = []
    losses = []
    for seed in range(n_ensemble):
        model = MLPRegressor(hidden_layer_sizes=(20, 20), activation='relu', solver='lbfgs', alpha=0.001, max_iter=5000, random_state=seed)
        model.fit(X_scaled, y_scaled)
        models.append(model)
        losses.append(model.loss_)
    nn_models[i] = models
    nn_x_scalers[i] = x_scaler
    nn_y_scalers[i] = y_scaler
    nn_training_losses[i] = losses

display(pd.DataFrame({"Function": list(nn_training_losses), "Mean training loss": [np.mean(nn_training_losses[i]) for i in nn_training_losses]}).set_index("Function").round(6))

,Mean training loss
Function,
1,0.000924
2,0.006460
3,0.000440
4,0.000198
5,0.000276
6,0.000301
7,0.000427
8,0.000392


### Neural-network LOOCV

In [27]:
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
import numpy as np
nn_loocv_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    nn_predictions = []
    baseline_predictions = []
    actual_values = []
    for j in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[j] = False
        X_train = X[mask]
        y_train = y[mask]
        X_test = X[j:j + 1]
        y_test = y[j]
        x_scaler = StandardScaler()
        y_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(X_train)
        X_test_scaled = x_scaler.transform(X_test)
        y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
        model = MLPRegressor(hidden_layer_sizes=(20, 20), activation='relu', solver='lbfgs', alpha=0.001, max_iter=5000, random_state=42)
        model.fit(X_train_scaled, y_train_scaled)
        pred_scaled = model.predict(X_test_scaled)[0]
        pred = y_scaler.inverse_transform([[pred_scaled]])[0, 0]
        nn_predictions.append(pred)
        baseline_predictions.append(np.mean(y_train))
        actual_values.append(y_test)
    nn_mae = mean_absolute_error(actual_values, nn_predictions)
    baseline_mae = mean_absolute_error(actual_values, baseline_predictions)
    beats_baseline = nn_mae < baseline_mae
    nn_loocv_results[i] = {'nn_mae': nn_mae, 'baseline_mae': baseline_mae, 'beats_baseline': beats_baseline}

display(pd.DataFrame.from_dict(nn_loocv_results, orient="index").rename_axis("Function").round(6))

,nn_mae,baseline_mae,beats_baseline
Function,,,
1,0.001776,0.001111,False
2,0.218453,0.237181,True
3,0.060294,0.050129,False
4,1.906639,7.081622,True
5,224.182406,1689.428734,True
6,0.235898,0.497547,True
7,0.376668,0.586378,True
8,0.358344,0.941538,True


### Neural-network candidates

In [28]:
rng_nn = np.random.default_rng(42)
nn_candidates = {}
for i in range(1, 9):
    n_dimensions = data[i].shape[1] - 1
    candidates = rng_nn.random((100000, n_dimensions))
    candidates_scaled = nn_x_scalers[i].transform(candidates)
    ensemble_predictions_scaled = np.array([model.predict(candidates_scaled) for model in nn_models[i]])
    ensemble_predictions = np.array([nn_y_scalers[i].inverse_transform(preds.reshape(-1, 1)).ravel() for preds in ensemble_predictions_scaled])
    mean_predictions = ensemble_predictions.mean(axis=0)
    disagreement = ensemble_predictions.std(axis=0)
    idx = np.argmax(mean_predictions)
    selected_point = candidates[idx]
    predicted_output = mean_predictions[idx]
    model_disagreement = disagreement[idx]
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    best_index = np.argmax(y_existing)
    best_point = X_existing[best_index]
    distance_best = np.linalg.norm(selected_point - best_point)
    distances_existing = np.linalg.norm(X_existing - selected_point, axis=1)
    distance_nearest = distances_existing.min()
    nn_candidates[i] = {'point': selected_point, 'predicted': predicted_output, 'disagreement': model_disagreement, 'distance_best': distance_best, 'distance_nearest': distance_nearest}

### GP and NN comparison

In [29]:
gp_nn_distances = {}
for i in range(1, 9):
    nn_point = nn_candidates[i]['point']
    gp_nn_distances[i] = {'UCB': {}, 'EI': {}}
    for kappa in ucb_kappas:
        ucb_point = ucb_tuning_results[i][kappa]['point']
        distance = np.linalg.norm(nn_point - ucb_point)
        gp_nn_distances[i]['UCB'][kappa] = distance
    for xi_prop in xi_proportions:
        ei_point = ei_tuning_results[i][xi_prop]['point']
        distance = np.linalg.norm(nn_point - ei_point)
        gp_nn_distances[i]['EI'][xi_prop] = distance

### Candidate neighbourhoods

In [30]:
import numpy as np
ucb_settings = [0.5, 1.5, 2.5]
ei_settings = [0.0, 0.05, 0.1]
neighbourhood_results = {}
for i in range(1, 9):
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    neighbourhood_results[i] = {}
    for kappa in ucb_settings:
        point = ucb_tuning_results[i][kappa]['point']
        distances = np.linalg.norm(X_existing - point, axis=1)
        nearest_idx = np.argsort(distances)[:3]
        key = f'UCB kappa={kappa}'
        neighbourhood_results[i][key] = []
        for idx in nearest_idx:
            neighbourhood_results[i][key].append({'distance': distances[idx], 'output': y_existing[idx]})
    for xi_prop in ei_settings:
        point = ei_tuning_results[i][xi_prop]['point']
        distances = np.linalg.norm(X_existing - point, axis=1)
        nearest_idx = np.argsort(distances)[:3]
        key = f'EI xi={xi_prop}'
        neighbourhood_results[i][key] = []
        for idx in nearest_idx:
            neighbourhood_results[i][key].append({'distance': distances[idx], 'output': y_existing[idx]})
    point = nn_candidates[i]['point']
    distances = np.linalg.norm(X_existing - point, axis=1)
    nearest_idx = np.argsort(distances)[:3]
    neighbourhood_results[i]['NN'] = []
    for idx in nearest_idx:
        neighbourhood_results[i]['NN'].append({'distance': distances[idx], 'output': y_existing[idx]})

### Function 5 local refinement

In [31]:
i = 5
X_existing = data[i].drop(columns='output').values.astype(float)
y_existing = data[i]['output'].values.astype(float)
best_idx = np.argmax(y_existing)
best_point = X_existing[best_idx]
best_output = y_existing[best_idx]
rng_local = np.random.default_rng(42)
n_local = 100000
local_candidates = best_point + rng_local.normal(loc=0.0, scale=0.03, size=(n_local, len(best_point)))
local_candidates = np.clip(local_candidates, 0, 1)
dist_existing = np.min(np.linalg.norm(local_candidates[:, None, :] - X_existing[None, :, :], axis=2), axis=1)
valid = dist_existing > 0.005
local_candidates = local_candidates[valid]
dist_existing = dist_existing[valid]
gp_mean, gp_std = gp_models[i].predict(local_candidates, return_std=True)
local_scaled = nn_x_scalers[i].transform(local_candidates)
nn_preds_scaled = np.array([model.predict(local_scaled) for model in nn_models[i]])
nn_preds = np.array([nn_y_scalers[i].inverse_transform(p.reshape(-1, 1)).ravel() for p in nn_preds_scaled])
nn_mean = nn_preds.mean(axis=0)
nn_disagreement = nn_preds.std(axis=0)
gp_rank = np.argsort(np.argsort(-gp_mean))
nn_rank = np.argsort(np.argsort(-nn_mean))
combined_rank = gp_rank + nn_rank
idx = np.argmin(combined_rank)
f5_local_candidate = local_candidates[idx]

print("Function 5 local candidate:", np.round(f5_local_candidate, 6))

Function 5 local candidate: [1. 1. 1. 1.]


### Final validation and submission

In [32]:
final_candidates = {1: np.array([0.858598, 0.697368]), 2: np.array([0.706079, 0.15803]), 3: np.array([0.45032, 0.561567, 0.490526]), 4: np.array([0.384332, 0.391111, 0.336887, 0.416488]), 5: np.array([1.0, 1.0, 1.0, 1.0]), 6: np.array([0.387946, 0.435172, 0.345126, 0.743513, 0.076825]), 7: np.array([0.054205, 0.21683, 0.421532, 0.346756, 0.385001, 0.543902]), 8: np.array([0.030327, 0.231267, 0.050266, 0.208295, 0.81852, 0.414744, 0.307096, 0.583908])}
all_valid = True
for i in range(1, 9):
    point = final_candidates[i]
    X_existing = data[i].drop(columns='output').values.astype(float)
    expected_dimensions = X_existing.shape[1]
    correct_dimensions = len(point) == expected_dimensions
    in_bounds = np.all((point >= 0) & (point <= 1))
    distances = np.linalg.norm(X_existing - point, axis=1)
    nearest_distance = distances.min()
    duplicate = np.any(np.all(np.isclose(X_existing, point, atol=1e-10, rtol=0), axis=1))
    valid = correct_dimensions and in_bounds and (not duplicate)
    all_valid = all_valid and valid
for i in range(1, 9):
    submission = '-'.join((f'{x:.6f}' for x in final_candidates[i]))

print("All eight points valid:", all_valid)
print("\nWEEK 7 SUBMISSION POINTS")
for i, point in final_candidates.items():
    print(f"Function {i}: " + "-".join(f"{x:.6f}" for x in point))
if not all_valid:
    raise ValueError("One or more final points failed validation.")

All eight points valid: True

WEEK 7 SUBMISSION POINTS
Function 1: 0.858598-0.697368
Function 2: 0.706079-0.158030
Function 3: 0.450320-0.561567-0.490526
Function 4: 0.384332-0.391111-0.336887-0.416488
Function 5: 1.000000-1.000000-1.000000-1.000000
Function 6: 0.387946-0.435172-0.345126-0.743513-0.076825
Function 7: 0.054205-0.216830-0.421532-0.346756-0.385001-0.543902
Function 8: 0.030327-0.231267-0.050266-0.208295-0.818520-0.414744-0.307096-0.583908
